# 12 · Defence: the fix, the action, and a WAF rule

The proposal's last task: for every attacker, suggest **how to fix** the weakness and **how to block** the attacker (`ml/defense.py`).

1. **Fix: k-NN over a knowledge base.** `ml/knowledge_base.py` has 16 weaknesses (SQL injection in a query, blind SQL injection, stored XSS, password spraying, ID enumeration, exposed files, ...), each with its OWASP fix. For every attack step of a session, the evidence (pages, fields, payload words, server answers) is turned into words and the nearest entries are returned.
2. **Action: Decision Tree.** Block the IP, rate-limit it, or allow it, learned from session numbers. Labels come from the TRUE kind of each session: payload attacks -> block; brute force / enumeration / scanning only -> rate limit; normal -> allow.
3. **WAF rules:** ModSecurity and Nginx lines built from the evidence (IP, page, field).

Data: development runs only (run3, run4 and camp7, both sites). The locked runs are not used.

**Honest notes:** the knowledge base is ours (written from OWASP), so k-NN retrieves from our own list, not from past real incidents. The generator labels only one kind of sub-type (fast / slow / spray brute force), so that is the only place where the k-NN's choice inside a family can be checked.

In [1]:
import sys, re
from pathlib import Path
import pandas as pd

ROOT = Path.cwd() if (Path.cwd() / 'data').exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
from sklearn.tree import DecisionTreeClassifier, export_text
from eval.final_test import run
from ml.stage import add_stages
from ml.knowledge_base import KNOWLEDGE_BASE
from ml.defense import (FEATURES, FixFinder, add_defense, evidence_text, session_features, step_rows,
                        _TRAVERSAL, _OPERATOR)
from ml.train_defense import DEV_RUNS, TREE, true_actions

D = ROOT / 'data'
NAMES = ['run3 JS', 'run4 JS', 'run4 DVWA', 'camp7 JS', 'camp7 DVWA']
runs = {}
for name, (log, labels) in zip(NAMES, DEV_RUNS):
    req, sessions, report = run(D / 'raw' / log, D / 'labels' / labels)
    req, sessions = add_stages(req, sessions)
    lab = pd.read_csv(D / 'labels' / labels)
    variant = lab[lab['variant'].notna()].groupby('ip')['variant'].agg(lambda s: s.mode()[0])
    runs[name] = dict(req=req, sessions=sessions, variant=variant)
    print(name, report)

run3 JS {'log_lines': 4397, 'labels': 4397, 'matched': 4397, 'unmatched_log_lines': 0, 'unmatched_labels': 0}


run4 JS {'log_lines': 4296, 'labels': 4296, 'matched': 4296, 'unmatched_log_lines': 0, 'unmatched_labels': 0}


run4 DVWA {'log_lines': 3586, 'labels': 3567, 'matched': 3567, 'unmatched_log_lines': 19, 'unmatched_labels': 0}


camp7 JS {'log_lines': 6372, 'labels': 6372, 'matched': 6372, 'unmatched_log_lines': 0, 'unmatched_labels': 0}


camp7 DVWA {'log_lines': 6456, 'labels': 6387, 'matched': 6387, 'unmatched_log_lines': 69, 'unmatched_labels': 0}


## 1. The knowledge base

In [2]:
pd.set_option('display.max_colwidth', 70)
kb = pd.DataFrame([{'id': e['id'], 'family': e['family'], 'weakness': e['title'], 'OWASP': e['owasp'],
                    'CWE': e['cwe'], 'first fix': e['fix'][0]} for e in KNOWLEDGE_BASE])
display(kb)
print(len(kb), 'entries,', kb['family'].nunique(), 'families')

,id,family,weakness,OWASP,CWE,first fix
0,sqli-query,sqli,SQL injection in a query parameter,A03:2021 Injection,CWE-89,Build every SQL query with parameterised statements (prepared stat...
1,sqli-blind,sqli,Blind (time-based) SQL injection,A03:2021 Injection,CWE-89,Use parameterised statements for every query (the same fix as any ...
2,sqli-login,sqli,SQL injection in the login form (authentication bypass),A03:2021 Injection,CWE-89,Look the user up with a parameterised query and compare a password...
3,xss-reflected,xss,Reflected cross-site scripting (XSS),A03:2021 Injection,CWE-79,"Encode every user value for the place it is written (HTML body, at..."
4,xss-stored,xss,Stored cross-site scripting in user content,A03:2021 Injection,CWE-79,"Encode user content when it is shown, on every page that shows it."
5,traversal-file,traversal,Path traversal / local file inclusion,A01:2021 Broken Access Control,CWE-22,Never build a file path from user input; map an allowed name or ID...
6,traversal-encoded,traversal,Encoded path traversal (filter bypass),A01:2021 Broken Access Control,CWE-22,"Decode the input fully (once, to its canonical form) before any ch..."
7,cmdi-os,cmdi,OS command injection,A03:2021 Injection,CWE-78,Do not call the system shell with user input; use the language's l...
8,cmdi-blind,cmdi,Blind (time-based) command injection,A03:2021 Injection,CWE-78,Remove the shell call or pass arguments as a list without a shell.
9,bruteforce-account,bruteforce,Password guessing on one account,A07:2021 Identification and Authentication Failures,CWE-307,"Limit failed logins per account: slow down, then lock for a short ..."


16 entries, 7 families


## 2. k-NN: does it pick the right fix?

Every attack step the system found (detector + session layer + attack type) is one query. Two checks:

* **Family:** the step's family is set by the detection pipeline, and k-NN looks only inside that family. How often is the family the true one (the labels of the step's requests)?
* **Sub-type (brute force):** fast guessing and slow guessing on one account should get the one-account or slow entry, and spraying many accounts the spraying entry.

In [3]:
finder = FixFinder()
rows = []
for name, r in runs.items():
    for sid, steps in step_rows(r['req']).items():
        for step, g in steps.items():
            truth = g['attack_type'].replace('', 'normal').mode()[0].replace('enumeration', 'enum')
            text = evidence_text(step, g)
            rows.append({'run': name, 'session': sid, 'step': step, 'truth': truth,
                         'variant': r['variant'].get(sid.split('#')[0], '') if truth == 'bruteforce' else '',
                         'pick': finder.nearest(text, 1, family=step)[0][0]['id'],
                         'pick (evidence only)': (finder.nearest(evidence_text(step, g, with_step=False), 1) or [({'id': '', 'family': ''}, 0)])[0][0]})
steps = pd.DataFrame(rows)
steps['family (evidence only)'] = steps['pick (evidence only)'].map(lambda e: e['family'])
atk = steps[steps['truth'] != 'normal']
print('attack steps:', len(atk), '| steps made only of normal requests:', int((steps['truth'] == 'normal').sum()))
print('family right (pipeline family):', round((atk['step'] == atk['truth']).mean(), 3))
print('family right (k-NN on evidence alone, all 16 entries):', round((atk['family (evidence only)'] == atk['truth']).mean(), 3))
display(atk.assign(ok=atk['step'] == atk['truth']).groupby('truth')['ok'].mean().round(3).to_frame('family right'))

attack steps: 822 | steps made only of normal requests: 38
family right (pipeline family): 0.91
family right (k-NN on evidence alone, all 16 entries): 0.781


,family right
truth,
bruteforce,1.000
cmdi,0.843
enum,0.991
scan,1.000
sqli,0.742
traversal,1.000
xss,0.915


In [4]:
bf = steps[(steps['truth'] == 'bruteforce') & (steps['variant'] != '') & (steps['step'] == 'bruteforce')]
display(pd.crosstab(bf['variant'], bf['pick']))
right = {'fast': 'bruteforce-account', 'slow': 'bruteforce-slow', 'spray': 'bruteforce-spray'}
print('brute-force sub-type right:', int((bf['pick'] == bf['variant'].map(right)).sum()), 'of', len(bf))

pick,bruteforce-account,bruteforce-slow,bruteforce-spray
variant,,,
fast,42,0,1
slow,0,15,0
spray,1,0,22


brute-force sub-type right: 79 of 81


**Sub-type (payload attacks).** The dev runs were made before the generator labelled payload sub-types, so here the sub-type of each request is derived from the request with the same rule the generator now uses (`payload_variant` in `lab/generator/generate_traffic.py`): SQL injection blind (time words) / in the login form / in a query; XSS stored (feedback, registration) / reflected; traversal encoded / plain; command injection blind / OS. One attacker usually mixes sub-types, so three numbers: top-1 = the most common sub-type of the step; top-1 is one of the sub-types the step really used; top-2 contains the most common one. run6 will carry the generator's own labels.

In [5]:
from urllib.parse import unquote_plus
from ml.defense import _TIME_WORDS
ENTRY = {('sqli', 'blind'): 'sqli-blind', ('sqli', 'login'): 'sqli-login', ('sqli', 'query'): 'sqli-query',
         ('xss', 'stored'): 'xss-stored', ('xss', 'reflected'): 'xss-reflected',
         ('traversal', 'encoded'): 'traversal-encoded', ('traversal', 'plain'): 'traversal-file',
         ('cmdi', 'blind'): 'cmdi-blind', ('cmdi', 'os'): 'cmdi-os'}
def sub_type(kind, method, path, body):
    v = unquote_plus(str(path).partition('?')[2]) + ' ' + unquote_plus(str(body))
    page = str(path).split('?')[0]
    if kind == 'sqli': return 'blind' if _TIME_WORDS.search(v) else 'login' if page == '/rest/user/login' else 'query'
    if kind == 'xss': return 'stored' if method == 'POST' and page in ('/api/Feedbacks/', '/api/Users/') else 'reflected'
    if kind == 'traversal': return 'encoded' if re.search(r'%25|%2e', str(path), re.I) else 'plain'
    return 'blind' if _TIME_WORDS.search(v) else 'os'

rows = []
for name, r in runs.items():
    for sid, st in step_rows(r['req']).items():
        for kind, g in st.items():
            real = g[g['attack_type'] == kind]
            if kind not in ('sqli', 'xss', 'traversal', 'cmdi') or real.empty:
                continue
            subs = [ENTRY[(kind, sub_type(kind, m, p, b))] for m, p, b in zip(real['method'], real['path'], real['body'])]
            near = [e['id'] for e, _ in finder.nearest(evidence_text(kind, g), 2, family=kind)]
            main = pd.Series(subs).mode()[0]
            rows.append({'family': kind, 'top-1 = most common': near[0] == main,
                         'top-1 used by the step': near[0] in subs, 'top-2 has most common': main in near})
subs_df = pd.DataFrame(rows)
display(subs_df.groupby('family').mean().round(3))
print('all payload steps:', len(subs_df), subs_df.drop(columns='family').mean().round(3).to_dict())

,top-1 = most common,top-1 used by the step,top-2 has most common
family,,,
cmdi,0.581,1.000,1.000
sqli,0.621,0.864,0.947
traversal,1.000,1.000,1.000
xss,0.873,0.992,1.000


all payload steps: 437 {'top-1 = most common': 0.769, 'top-1 used by the step': 0.957, 'top-2 has most common': 0.984}


In [6]:
print('Which entry k-NN picks, per step family:')
display(steps.groupby(['step', 'pick']).size().to_frame('steps'))

Which entry k-NN picks, per step family:


steps
step       pick                     
bruteforce bruteforce-account     59
           bruteforce-slow        17
           bruteforce-spray       43
cmdi       cmdi-blind             51
           cmdi-os                70
enum       enum-idor             113
scan       scan-admin             77
           scan-dirs              19
           scan-files             22
sqli       sqli-blind             27
           sqli-login             29
           sqli-query            102
traversal  traversal-encoded     102
xss        xss-reflected         119
           xss-stored             10

## 3. Decision Tree: block, rate-limit or allow

Trained on one site, tested on the other (the tree never saw that site). Baseline: the action from our risk level alone (critical/high -> block, medium -> rate limit, else allow).

In [7]:
def table(names):
    parts = []
    for n in names:
        r = runs[n]
        x = session_features(r['sessions'], r['req'])
        x['action'] = true_actions(r['req'], r['sessions']).values
        x['baseline'] = r['sessions']['risk_level'].map({'critical': 'block', 'high': 'block', 'medium': 'rate_limit'}).fillna('allow').values
        parts.append(x)
    return pd.concat(parts)

JS, DV = ['run3 JS', 'run4 JS', 'camp7 JS'], ['run4 DVWA', 'camp7 DVWA']
res = []
for train, test, label in [(JS, DV, 'Juice Shop -> DVWA'), (DV, JS, 'DVWA -> Juice Shop')]:
    a, b = table(train), table(test)
    tree = DecisionTreeClassifier(**TREE).fit(a[FEATURES], a['action'])
    pred = tree.predict(b[FEATURES])
    res.append({'train -> test': label, 'sessions': len(b), 'tree accuracy': round((pred == b['action']).mean(), 3),
                'risk-level baseline': round((b['baseline'] == b['action']).mean(), 3),
                'normal sessions blocked (tree)': int(((pred == 'block') & (b['action'] == 'allow')).sum()),
                'attackers allowed (tree)': int(((pred == 'allow') & (b['action'] != 'allow')).sum())})
    print(label); display(pd.crosstab(b['action'], pred, rownames=['true'], colnames=['tree']))
display(pd.DataFrame(res).set_index('train -> test'))

Juice Shop -> DVWA


tree,allow,block,rate_limit
true,,,
allow,420,0,0
block,0,127,2
rate_limit,0,2,49


DVWA -> Juice Shop


tree,allow,block,rate_limit
true,,,
allow,630,0,0
block,0,178,0
rate_limit,0,2,90


,sessions,tree accuracy,risk-level baseline,normal sessions blocked (tree),attackers allowed (tree)
train -> test,,,,,
Juice Shop -> DVWA,600,0.993,0.825,0,0
DVWA -> Juice Shop,900,0.998,0.874,0,0


The tree used by the system is trained on all development runs (`python -m ml.train_defense`). Its block rules, as a person can read them:

In [8]:
allruns = table(NAMES)
final_tree = DecisionTreeClassifier(**TREE).fit(allruns[FEATURES], allruns['action'])
print(export_text(final_tree, feature_names=FEATURES))
imp = pd.Series(final_tree.feature_importances_, index=FEATURES).sort_values(ascending=False)
display(imp[imp > 0].round(3).to_frame('importance'))

|--- risk_score <= 20.00
|   |--- class: allow
|--- risk_score >  20.00
|   |--- attack_requests <= 1.50
|   |   |--- class: rate_limit
|   |--- attack_requests >  1.50
|   |   |--- stage_rank <= 1.50
|   |   |   |--- class: rate_limit
|   |   |--- stage_rank >  1.50
|   |   |   |--- failed_logins <= 11.50
|   |   |   |   |--- class: block
|   |   |   |--- failed_logins >  11.50
|   |   |   |   |--- class: block



,importance
risk_score,0.511
attack_requests,0.246
stage_rank,0.237
failed_logins,0.006


**One fixed rule above the tree.** The tree decides from the kind of attack, so an ID enumeration that really took data (a *critical* session: high risk and the attack reached exploitation or data theft) would only be rate-limited. A critical session is therefore always blocked. How often does that rule change the tree's answer on the dev runs, and does it ever block a normal visitor?

In [9]:
rows = []
for name, r in runs.items():
    o = add_defense(r['req'], r['sessions'], finder, final_tree)
    truth = true_actions(r['req'], r['sessions']).values
    over = o['action_reason'].str.startswith('critical')
    rows.append({'run': name, 'critical sessions': int((o['risk_level'] == 'critical').sum()),
                 'changed to block by the rule': int(over.sum()),
                 'of them true payload attackers': int((over & (truth == 'block')).sum()),
                 'of them brute force / enum / scan': int((over & (truth == 'rate_limit')).sum()),
                 'normal visitors blocked': int(((o['action'] == 'block') & (truth == 'allow')).sum())})
display(pd.DataFrame(rows).set_index('run'))

,critical sessions,changed to block by the rule,of them true payload attackers,of them brute force / enum / scan,normal visitors blocked
run,,,,,
run3 JS,5,5,0,5,0
run4 JS,1,0,0,0,0
run4 DVWA,0,0,0,0,0
camp7 JS,35,2,0,2,0
camp7 DVWA,30,2,0,2,0


## 4. WAF rules

Examples of what the system writes for one blocked and one rate-limited attacker (camp7, Juice Shop).

In [10]:
r = runs['camp7 JS']
out = add_defense(r['req'], r['sessions'], finder, final_tree)
for action in ['block', 'rate_limit']:
    s = out[out['action'] == action].iloc[0]
    print(f"=== {s['ip']}  action={action}  risk={s['risk_score']} ({s['risk_level']})  stage={s['stage']}")
    for f in s['fixes']:
        m = f['matches'][0]
        print(f"  {f['step']}: {m['title']} [{m['owasp']}, {m['cwe']}] -> {m['fix'][0]}")
    print('\n'.join('  ' + line for line in s['waf_rules']))
    print()

=== 10.1.0.10  action=block  risk=100 (high)  stage=intrusion
  cmdi: OS command injection [A03:2021 Injection, CWE-78] -> Do not call the system shell with user input; use the language's library function for the job instead (e.g. a DNS or ping library).
  scan: Probing for exposed sensitive files [A05:2021 Security Misconfiguration, CWE-538] -> Do not deploy source control folders, backups or config files to the web root.
  sqli: SQL injection in a query parameter [A03:2021 Injection, CWE-89] -> Build every SQL query with parameterised statements (prepared statements); never join user input into the query text.
  traversal: Encoded path traversal (filter bypass) [A01:2021 Broken Access Control, CWE-22] -> Decode the input fully (once, to its canonical form) before any check.
  SecRule REMOTE_ADDR "@ipMatch 10.1.0.10" "id:1000001,phase:1,deny,status:403,log,msg:'AI Attack Investigator: block attacker 10.1.0.10'"
  # nginx: deny 10.1.0.10;
  SecRule REQUEST_FILENAME "@streq /rest/produc

Two of the rule patterns are plain regular expressions (command injection and path traversal), so we can check them on the dev logs: how many attack requests of that kind they match, and how many normal requests they would wrongly block. (The SQL injection and XSS rules use ModSecurity's own `@detectSQLi` / `@detectXSS`, which cannot be run here.)

In [11]:
from urllib.parse import unquote, unquote_plus
checks = {'cmdi': re.compile(_OPERATOR['cmdi'].split(' ', 1)[1]),
          'traversal': re.compile(_TRAVERSAL.split(' ', 1)[1])}
all_req = pd.concat([r['req'] for r in runs.values()])
def as_waf_sees(df, kind):
    t = df['path'].astype(str) + ' ' + df['body'].astype(str)
    if kind == 'traversal':          # this rule reads the raw URI, still encoded
        return t
    # ModSecurity decodes ARGS once, then the rule's t:urlDecodeUni x2 and t:removeComments
    return t.map(lambda x: re.sub(r'/\*.*?\*/', '', unquote(unquote(unquote_plus(x)))))
rows = []
norm = all_req[all_req['label'] == 'normal']
for kind, rx in checks.items():
    atk = all_req[all_req['attack_type'] == kind]
    rows.append({'rule': kind, 'attack requests': len(atk),
                 'matched': round(as_waf_sees(atk, kind).str.contains(rx).mean(), 3),
                 'normal requests matched (would be blocked)': int(as_waf_sees(norm, kind).str.contains(rx).sum()),
                 'normal requests': len(norm)})
display(pd.DataFrame(rows).set_index('rule'))

,attack requests,matched,normal requests matched (would be blocked),normal requests
rule,,,,
cmdi,526,0.882,0,16176
traversal,586,1.000,0,16176


**What blocking the IP would save.** If the IP had been blocked right after its first request the detector flagged, how many of that attacker's later attack requests would never have reached the site?

In [12]:
rows = []
for name, r in runs.items():
    o = add_defense(r['req'], r['sessions'], finder, final_tree)
    blocked = set(o.loc[o['action'] == 'block', 'session'])
    q = r['req'][r['req']['session'].isin(blocked)].sort_values(['session', 'timestamp'], kind='stable')
    first = q[q['pred'] == 1].groupby('session')['timestamp'].min()
    after = q[(q['attack_type'] != '') & (q['timestamp'] > q['session'].map(first))]
    total = q[q['attack_type'] != '']
    rows.append({'run': name, 'blocked attackers': len(blocked), 'their attack requests': len(total),
                 'stopped by the block': len(after), 'share stopped': round(len(after) / max(len(total), 1), 3)})
display(pd.DataFrame(rows).set_index('run'))

,blocked attackers,their attack requests,stopped by the block,share stopped
run,,,,
run3 JS,54,411,187,0.455
run4 JS,52,310,217,0.700
run4 DVWA,52,294,193,0.656
camp7 JS,81,2751,1896,0.689
camp7 DVWA,81,2720,1873,0.689


## 5. Conclusions

* **Knowledge base:** 16 weaknesses in 7 families, each with its OWASP fix, Top 10 category and CWE.
* **Fix (k-NN):** the family of an attack step is right for 91% of the 822 attack steps (it comes from the detection pipeline). k-NN on the evidence alone, over all 16 entries, gets the family right less often, 78% of the time, which is why the system lets the detector choose the family and k-NN choose the weakness inside it. Inside the brute-force family the sub-type is right for 79 of 81 attackers (one account / slow / spraying). For payload attacks (sub-type derived from the request, see section 2) the first pick is a sub-type the attacker really used in 95.7% of steps, and the right one is in the top two in 98.4%. One attacker mixes sub-types, so the first pick alone matches the most common one less often (76.9%).
* **Action (Decision Tree):** trained on one site and tested on the other, the tree gives the right action to 99.3% (Juice Shop -> DVWA) and 99.8% (DVWA -> Juice Shop) of sessions, against 82.5% / 87.4% for the risk-level baseline, and blocks **no** normal visitor and lets **no** attacker through. Two features were removed on the dev runs because they described the site, not the attacker (share of attack requests, number of login attempts). The tree learns the action WE defined for each kind of session, which is said in the report.
* **Critical = block:** a fixed rule above the tree blocks every critical session (the attack worked); on the dev runs it changed the tree's answer for 9 sessions, all brute force / enumeration that worked, and blocked no normal visitor.
* **WAF rules:** the command injection and path traversal rules match 88% and 100% of those attacks in the dev logs and would block 0 of 16,176 normal requests. The SQL injection and XSS rules use ModSecurity's own detectors and are not tested here.
* **Effect:** blocking an attacker's IP right after its first detected request would have stopped 65-72% of its later attack requests.
* These are development numbers. The final check of the defence is the locked run6.